In [ ]:
%pip -q install qwen-tts fastapi uvicorn soundfile

In [ ]:
import base64, hashlib, io, os, secrets, threading
from collections import deque
import soundfile as sf
import torch
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel, Field
from qwen_tts import Qwen3TTSModel

if not torch.cuda.is_available():
    raise RuntimeError("请在 Kaggle 设置中启用 GPU")
model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-Base", device_map="cuda:0",
    dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)
from kaggle_secrets import UserSecretsClient
try:
    token = UserSecretsClient().get_secret("KAGGLE_TTS_TOKEN")
except Exception:
    token = None
token = token or os.environ.get("KAGGLE_TTS_TOKEN") or secrets.token_urlsafe(24)
app = FastAPI(docs_url=None, redoc_url=None)
gpu_lock = threading.Lock()
prompt_cache = {}
jobs_lock = threading.Lock()
jobs = {}
completed_jobs = deque()

class Request(BaseModel):
    text: list[str] = Field(min_length=1, max_length=64)
    mode: str
    ref_audio_base64: str = Field(max_length=16_000_000)
    ref_text: str
    language: str = "Chinese"
    temperature: float = Field(default=.7, gt=0, le=2)

def authorize(authorization):
    if not secrets.compare_digest(authorization.encode(), ("Bearer " + token).encode()):
        raise HTTPException(401)

def validate_request(request, authorization):
    authorize(authorization)
    if request.mode != "clone":
        raise HTTPException(400, "需要 clone 模式")
    if not request.ref_audio_base64:
        raise HTTPException(422, "参考音频无效")

def synthesize(request):
    key = hashlib.sha256((request.ref_audio_base64 + "|" + request.ref_text).encode()).hexdigest()
    with gpu_lock, torch.inference_mode():
        if key not in prompt_cache:
            try:
                audio = base64.b64decode(request.ref_audio_base64, validate=True)
                sample, rate = sf.read(io.BytesIO(audio), dtype="float32")
            except (ValueError, RuntimeError):
                raise HTTPException(422, "参考音频无效")
            prompt_cache[key] = model.create_voice_clone_prompt(
                ref_audio=(sample, rate), ref_text=request.ref_text,
            )
        waves, rate = model.generate_voice_clone(
            text=request.text, language=[request.language]*len(request.text),
            voice_clone_prompt=prompt_cache[key], temperature=request.temperature,
            max_new_tokens=min(640, max(180, max(len(t) for t in request.text) * 6)),
        )
    outputs = []
    for waveform in waves:
        buf = io.BytesIO()
        sf.write(buf, waveform, rate, format="WAV", subtype="PCM_16")
        outputs.append({"audio_base64": base64.b64encode(buf.getvalue()).decode("ascii")})
    return outputs

@app.post("/tts")
def tts(request: Request, authorization: str = Header(default="")):
    validate_request(request, authorization)
    return {"outputs": synthesize(request)}

def complete_job(job_id, request):
    try:
        result = {"status": "done", "outputs": synthesize(request)}
    except Exception as exc:
        result = {"status": "error", "detail": str(exc)}
    with jobs_lock:
        jobs[job_id] = result
        completed_jobs.append(job_id)
        if len(completed_jobs) > 32:
            del jobs[completed_jobs.popleft()]

@app.post("/tts_async")
def tts_async(request: Request, authorization: str = Header(default="")):
    validate_request(request, authorization)
    with jobs_lock:
        job_id = secrets.token_hex(8)
        jobs[job_id] = {"status": "running"}
    worker = threading.Thread(target=complete_job, args=(job_id, request), daemon=True)
    try:
        worker.start()
    except RuntimeError:
        with jobs_lock:
            del jobs[job_id]
        raise
    return {"job_id": job_id, "status": "running"}

@app.get("/tts_job/{job_id}")
def tts_job(job_id: str, authorization: str = Header(default="")):
    authorize(authorization)
    with jobs_lock:
        result = jobs.get(job_id)
    if result is None:
        raise HTTPException(404, "任务不存在或已过期")
    return result


In [ ]:
import pathlib, platform, re, socket, subprocess, time, urllib.request
import uvicorn
binary = pathlib.Path("/kaggle/working/cloudflared")
if not binary.exists():
    arch = {"x86_64":"amd64", "aarch64":"arm64"}[platform.machine()]
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-"+arch, binary)
    binary.chmod(0o700)
subprocess.run(["pkill", "-f", "cloudflared"], check=False)
with socket.socket() as s:
    s.bind(("127.0.0.1", 0))
    port = s.getsockname()[1]
server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=port, access_log=False))
threading.Thread(target=server.run, daemon=True).start()
log_path = pathlib.Path("/kaggle/working/cloudflared.log")
log_path.unlink(missing_ok=True)
subprocess.Popen([str(binary), "tunnel", "--url", f"http://127.0.0.1:{port}", "--logfile", str(log_path)], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(60):
    txt = log_path.read_text(errors="ignore") if log_path.exists() else ""
    match = re.search(r"https://[a-z0-9-]+[.]trycloudflare[.]com", txt)
    if match:
        print("KAGGLE_TTS_ENDPOINT_URL=" + match.group(0) + "/tts")
        print("KAGGLE_TTS_TOKEN=" + token)
        break
    time.sleep(0.5)
else:
    raise RuntimeError("隧道未建立")